In [1]:
import wrds
import pandas as pd
import numpy as np
db = wrds.Connection()  

WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


# Part 1: Define Top 40 Markets

- We will query headlines data from 07/01/2025 to 12/31/2025.  
- Top by market cap: price * shares outstandings. we use 06/30/25.

In [2]:
valdate = "2025-06-30"

sql_top40_ds = """
select
    m.infocode,
    m.valdate,
    m.consolmktval,
    s.dsseccode,
    s.dssctycode,
    s.dssecname,
    s.primexchmnem,
    s.ibesticker,
    c.dscmpyname,
    c.cmpyctrycode
from tr_ds_equities.ds2mktval m
join tr_ds_equities.ds2security s
  on m.infocode = s.primqtinfocode
join tr_ds_equities.ds2company c
  on s.dscmpycode = c.dscmpycode
where m.valdate = %(valdate)s
  and c.cmpyctrycode = 'US'
  and s.primexchmnem in ('NAS','NYS')
  and m.consolmktval is not null
  and upper(s.dssecname) not like '%%ETF%%'
  and upper(s.dssecname) not like '%%FUND%%'
  and upper(s.dssecname) not like '%%TRUST%%'
  and upper(s.dssecname) not like '%%ETN%%'
order by m.consolmktval desc
limit 40
"""

top40 = db.raw_sql(sql_top40_ds, params={"valdate": valdate})



In [3]:
top40.to_csv(f"top40_market_{valdate}.csv", index=False)

In [4]:
top40 = pd.read_csv(f"top40_market_{valdate}.csv")

# Part 2: Get Daily Open/Close Prices

We have 128 trading days between 2025-07-01 and 2025-12-31. This yields 3400 by 4 table

In [11]:
months = [ ("2025-07-01","2025-08-01"),
           ("2025-08-01","2025-09-01"),
          ("2025-09-01","2025-10-01"),
          ("2025-10-01","2025-11-01"),
          ("2025-11-01","2025-12-01"),
          ("2025-12-01","2026-01-01")]

codes = top40["infocode"].astype(int).tolist()

def chunks(lst, k=10):
    for i in range(0, len(lst), k):
        yield lst[i:i+k]

def pull_prices(start, end, infocodes):
    sql = """
    select infocode, marketdate, open_ as open, close_ as close
    from tr_ds_equities.ds2primqtprc
    where marketdate >= %(start)s
      and marketdate <  %(end)s
      and infocode in %(infocodes)s
    """
    return db.raw_sql(sql, params={"start": start, "end": end, "infocodes": tuple(infocodes)})

parts = []
for batch_i, batch in enumerate(chunks(codes, 10), start=1):
    for start, end in months:
        print(f"Batch {batch_i} ({batch[0]}..{batch[-1]}), {start} -> {end}")
        df = pull_prices(start, end, batch)          # if this errors, it stops—no loops
        #df.to_csv(f"ckpt_prices_b{batch_i}_{start}.csv", index=False)  # checkpoint
        parts.append(df)

prices = pd.concat(parts, ignore_index=True)
prices.to_csv("top40_marketdata_open_close_july_dec_2025.csv", index=False)


Batch 1 (66821..64084), 2025-07-01 -> 2025-08-01
Batch 1 (66821..64084), 2025-08-01 -> 2025-09-01
Batch 1 (66821..64084), 2025-09-01 -> 2025-10-01
Batch 1 (66821..64084), 2025-10-01 -> 2025-11-01
Batch 1 (66821..64084), 2025-11-01 -> 2025-12-01
Batch 1 (66821..64084), 2025-12-01 -> 2026-01-01
Batch 2 (46588..60529), 2025-07-01 -> 2025-08-01
Batch 2 (46588..60529), 2025-08-01 -> 2025-09-01
Batch 2 (46588..60529), 2025-09-01 -> 2025-10-01
Batch 2 (46588..60529), 2025-10-01 -> 2025-11-01
Batch 2 (46588..60529), 2025-11-01 -> 2025-12-01
Batch 2 (46588..60529), 2025-12-01 -> 2026-01-01
Batch 3 (67206..74139), 2025-07-01 -> 2025-08-01
Batch 3 (67206..74139), 2025-08-01 -> 2025-09-01
Batch 3 (67206..74139), 2025-09-01 -> 2025-10-01
Batch 3 (67206..74139), 2025-10-01 -> 2025-11-01
Batch 3 (67206..74139), 2025-11-01 -> 2025-12-01
Batch 3 (67206..74139), 2025-12-01 -> 2026-01-01
Batch 4 (60876..67773), 2025-07-01 -> 2025-08-01
Batch 4 (60876..67773), 2025-08-01 -> 2025-09-01
Batch 4 (60876..6777

In [12]:
prices = pd.read_csv("top40_marketdata_open_close_july_dec_2025.csv")

# Part 3: Get Headlines


## Get ISIN for 40 companies

In [16]:

valdate_dt = pd.to_datetime(valdate)

dsseccodes = tuple(top40["dsseccode"].astype(int).tolist())

isin_map = db.raw_sql("""
select dsseccode, startdate, enddate, isin, isin2
from tr_ds_equities.ds2isinchg
where dsseccode in %(codes)s
""", params={"codes": dsseccodes})

isin_map["startdate"] = pd.to_datetime(isin_map["startdate"])
isin_map["enddate"] = pd.to_datetime(isin_map["enddate"])

active_isin = isin_map[
    (isin_map["startdate"] <= valdate_dt) &
    (isin_map["enddate"] >= valdate_dt)
].copy()

active_isin["isin_final"] = active_isin["isin"].fillna(active_isin["isin2"])

top40_isin = top40.merge(
    active_isin[["dsseccode","isin_final"]].drop_duplicates("dsseccode"),
    on="dsseccode",
    how="left"
)

print("Missing ISIN:", top40_isin["isin_final"].isna().sum())
top40_isin[["dssecname","isin_final"]].head(10)


Missing ISIN: 0


,dssecname,isin_final
0,NVIDIA,US67066G1040
1,MICROSOFT,US5949181045
2,APPLE,US0378331005
3,AMAZON.COM,US0231351067
4,ALPHABET 'A',US02079K3059
5,META PLATFORMS A,US30303M1027
6,BROADCOM,US11135F1012
7,BERKSHIRE HATHAWAY 'A',US0846701086
8,TESLA,US88160R1014
9,JP MORGAN CHASE & CO.,US46625H1005


## Map ISIN to Raven Pack ID

In [17]:
isins = tuple(sorted(top40_isin["isin_final"].dropna().unique().tolist()))

rp_isin_map = db.raw_sql("""
select rp_entity_id, entity_name, isin
from ravenpack_common.wrds_rpa_company_mappings
where isin in %(isins)s
""", params={"isins": isins})

# one row per ISIN
rp_isin_map = rp_isin_map.drop_duplicates(subset=["isin"])

top40_rp = top40_isin.merge(
    rp_isin_map,
    left_on="isin_final",
    right_on="isin",
    how="left"
)

print("Missing rp_entity_id:", top40_rp["rp_entity_id"].isna().sum())


Missing rp_entity_id: 0


## Query News Headlines

In [18]:
import time

months = [ ("2025-07-01","2025-08-01"),
           ("2025-08-01","2025-09-01"),
          ("2025-09-01","2025-10-01"),
          ("2025-10-01","2025-11-01"),
          ("2025-11-01","2025-12-01"),
          ("2025-12-01","2026-01-01")]

rp_ids = sorted(top40_rp["rp_entity_id"].unique().tolist())

def chunks(lst, k=10):
    for i in range(0, len(lst), k):
        yield lst[i:i+k]

def pull_news_unique(start, end, batch):
    sql = """
    select distinct on (rp_entity_id, rp_story_id)
      rpa_date_utc,
      timestamp_utc,
      rp_entity_id,
      entity_name,
      headline,
      rp_story_id,
      source_name,
      rp_source_id,
      provider_id,
      provider_story_id,
      relevance,
      event_relevance,
      event_sentiment_score,
      topic,
      "group",
      type,
      sub_type,
      fact_level
    from ravenpack_dj.rpa_djpr_equities_2025
    where rpa_date_utc >= %(start)s
      and rpa_date_utc <  %(end)s
      and rp_entity_id in %(batch)s
    order by rp_entity_id, rp_story_id, timestamp_utc asc
    """
    return db.raw_sql(sql, params={"start": start, "end": end, "batch": tuple(batch)})

parts = []
for start, end in months:
    for bi, batch in enumerate(chunks(rp_ids, 10), start=1):
        print(f"{start}->{end} batch {bi}")
        df = pull_news_unique(start, end, batch)
        press_sources = {"PR Newswire","GlobeNewswire","Business Wire","DJ Global Press Release Wire"}
        df["is_press_release"] = df["source_name"].isin(press_sources)
        #df.to_csv(f"ckpt_headlines_{start}_b{bi}.csv", index=False)  # checkpoint
        parts.append(df)
        time.sleep(2)  # tiny pause to 

news = pd.concat(parts, ignore_index=True)
news.to_csv("top40_headlines_july_dec_2025.csv", index=False)
print("Saved rows:", len(news))


2025-07-01->2025-08-01 batch 1
2025-07-01->2025-08-01 batch 2
2025-07-01->2025-08-01 batch 3
2025-07-01->2025-08-01 batch 4
2025-08-01->2025-09-01 batch 1
2025-08-01->2025-09-01 batch 2
2025-08-01->2025-09-01 batch 3
2025-08-01->2025-09-01 batch 4
2025-09-01->2025-10-01 batch 1
2025-09-01->2025-10-01 batch 2
2025-09-01->2025-10-01 batch 3
2025-09-01->2025-10-01 batch 4
2025-10-01->2025-11-01 batch 1
2025-10-01->2025-11-01 batch 2
2025-10-01->2025-11-01 batch 3
2025-10-01->2025-11-01 batch 4
2025-11-01->2025-12-01 batch 1
2025-11-01->2025-12-01 batch 2
2025-11-01->2025-12-01 batch 3
2025-11-01->2025-12-01 batch 4
2025-12-01->2026-01-01 batch 1
2025-12-01->2026-01-01 batch 2
2025-12-01->2026-01-01 batch 3
2025-12-01->2026-01-01 batch 4
Saved rows: 494082


In [19]:
news = pd.read_csv("top40_headlines_july_dec_2025.csv")

## Link News to Returns via RP_entity_id + Finalize headlines

In [20]:
# Add rp entity id to prices
xwalk = (top40_rp[["infocode","dssecname","dscmpyname","isin_final","rp_entity_id","entity_name"]]
         .drop_duplicates("rp_entity_id"))
prices2 = prices.merge(xwalk[["infocode","rp_entity_id"]], on="infocode", how="left")
prices2.to_csv("top40_marketdata_open_close_july_dec_2025.csv", index=False)


In [21]:
# Fix Company Names by ENTITY ID
canon_name = top40_rp[["rp_entity_id","entity_name"]].drop_duplicates()
news = news.merge(canon_name, on="rp_entity_id", how="left", suffixes=("", "_canon"))
news.to_csv("top40_headlines_july_dec_2025.csv", index=False)
